# 2. Lake selection: 1,000 to 777

Documents how the manually labeled pool becomes the evaluation set reported in
the paper.

| Stage | Lakes | How |
| --- | --- | --- |
| Labeled | 1,000 | 250 per class across all six GrIS subregions |
| Visual QC | 793 | manual review of each lake's plotted time series |
| Final | 777 | evaluation set used in the paper |

The final set is distributed as an explicit manifest
(`data/processed/lake_manifest.csv`) rather than as a filter to re-derive, so
the split stays exactly reproducible.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "rpsgmm").is_dir():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Run this notebook from inside the repository.")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

print("Repository root:", REPO_ROOT)

In [ ]:
import pandas as pd

PROCESSED = REPO_ROOT / "data" / "processed"
manifest = pd.read_csv(PROCESSED / "lake_manifest.csv")
manifest.head()

## The selection chain

In [ ]:
stages = pd.DataFrame({
    "stage": ["labeled", "visual QC", "final (paper)"],
    "lakes": [len(manifest), int(manifest["in_visual_qc"].sum()), int(manifest["in_final_777"].sum())],
})
stages["dropped"] = stages["lakes"].shift(1).sub(stages["lakes"]).fillna(0).astype(int)
stages

In [ ]:
final = manifest[manifest["in_final_777"]]
print("Three-class balance (paper reports 189 / 392 / 196):")
print(final["label_3class"].value_counts().to_string())
print()
print("By region:")
print(final["region"].value_counts().to_string())

## The 16 lakes dropped after visual QC

These were set aside during manual review. They are listed here so the
composition of the evaluation set is fully transparent.

In [ ]:
dropped = manifest[manifest["in_visual_qc"] & ~manifest["in_final_777"]]
print(f"{len(dropped)} lakes")
dropped[["ids", "region", "label_4class", "area_m2", "elevation_m"]]

The manifest ships with the repository, so the split is fixed and does not
need to be re-derived.

---

Next: [`../02_rps_gmm/01_backscatter.ipynb`](../02_rps_gmm/01_backscatter.ipynb).